# Práctica · Spark DataFrames · Clientes y pedidos

Completa las celdas indicadas. Trabaja sobre el entorno dockerizado incluido en `spark_jupyter/`.

In [1]:
from iniciar_spark import get_spark
from pyspark.sql import functions as F

spark = get_spark("PracticaClientesPedidos")
print("SparkSession creada")
print("Versión de Spark:", spark.version)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/19 12:05:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession creada
Versión de Spark: 4.1.1


In [2]:
from pathlib import Path

data_dir = Path("/opt/spark-apps/datos")
clientes_path = data_dir / "clientes.csv"
pedidos_path = data_dir / "pedidos.csv"

print("Ruta clientes:", clientes_path)
print("Ruta pedidos:", pedidos_path)


Ruta clientes: /opt/spark-apps/datos/clientes.csv
Ruta pedidos: /opt/spark-apps/datos/pedidos.csv


## 1. Lee ambos CSV y muestra su esquema y algunas filas.

In [3]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

# Lectura con inferencia de esquema (los CSV usan ';' como separador)
clientes = (
    spark.read
    .option("header", True)
    .option("sep", ";")
    .option("inferSchema", True)
    .csv(str(clientes_path))
)

pedidos = (
    spark.read
    .option("header", True)
    .option("sep", ";")
    .option("inferSchema", True)
    .csv(str(pedidos_path))
)

print("=== Esquema clientes ===")
clientes.printSchema()
print(f"Total filas clientes: {clientes.count()}")
clientes.show(5, truncate=False)

print("=== Esquema pedidos ===")
pedidos.printSchema()
print(f"Total filas pedidos: {pedidos.count()}")
pedidos.show(5, truncate=False)


=== Esquema clientes ===
root
 |-- id_cliente: integer (nullable = true)
 |-- nombre: string (nullable = true)
 |-- ciudad: string (nullable = true)
 |-- segmento: string (nullable = true)

Total filas clientes: 43
+----------+------+----------+--------+
|id_cliente|nombre|ciudad    |segmento|
+----------+------+----------+--------+
|1         |Ana   | Sevilla  |Estandar|
|2         |Luis  |Bilbao    |Premium |
|3         |Marta | Alicante |Estandar|
|4         |Pablo | Madrid   |Premium |
|5         |Lucia |Bilbao    |Premium |
+----------+------+----------+--------+
only showing top 5 rows
=== Esquema pedidos ===
root
 |-- id_pedido: integer (nullable = true)
 |-- id_cliente: integer (nullable = true)
 |-- fecha: date (nullable = true)
 |-- producto: string (nullable = true)
 |-- cantidad: double (nullable = true)
 |-- precio_unitario: integer (nullable = true)

Total filas pedidos: 120
+---------+----------+----------+---------+--------+---------------+
|id_pedido|id_cliente|fecha  

## 2. Limpia el DataFrame de clientes: trim en ciudad y dropDuplicates.

In [4]:
# Trim a todas las columnas de texto y eliminacion de duplicados
clientes_limpio = (
    clientes
    .withColumn("nombre", F.trim(F.col("nombre")))
    .withColumn("ciudad", F.trim(F.col("ciudad")))
    .withColumn("segmento", F.trim(F.col("segmento")))
)

antes = clientes_limpio.count()
clientes_limpio = clientes_limpio.dropDuplicates(["id_cliente"])
despues = clientes_limpio.count()
print(f"Duplicados eliminados: {antes - despues}")

# Nulos por columna
print("=== Nulos por columna ===")
clientes_limpio.select(
    *[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in clientes_limpio.columns]
).show()

clientes_limpio.show(5, truncate=False)
print(f"Total clientes tras limpieza: {clientes_limpio.count()}")


Duplicados eliminados: 3
=== Nulos por columna ===
+----------+------+------+--------+
|id_cliente|nombre|ciudad|segmento|
+----------+------+------+--------+
|         0|     0|     0|       0|
+----------+------+------+--------+

+----------+------+--------+--------+
|id_cliente|nombre|ciudad  |segmento|
+----------+------+--------+--------+
|1         |Ana   |Sevilla |Estandar|
|2         |Luis  |Bilbao  |Premium |
|3         |Marta |Alicante|Estandar|
|4         |Pablo |Madrid  |Premium |
|5         |Lucia |Bilbao  |Premium |
+----------+------+--------+--------+
only showing top 5 rows
Total clientes tras limpieza: 40


## 3. Convierte tipos en pedidos y crea la columna `importe`.

In [5]:
# Gestion de nulos antes de transformar: descartamos pedidos sin cantidad o sin producto
pedidos_sin_nulos = pedidos.na.drop(subset=["cantidad", "producto"])

pedidos_tx = (
    pedidos_sin_nulos
    .withColumn("fecha", F.to_date("fecha", "yyyy-MM-dd"))
    .withColumn("cantidad", F.col("cantidad").cast("int"))
    .withColumn("precio_unitario", F.col("precio_unitario").cast("double"))
    .withColumn("producto", F.trim(F.col("producto")))
    .withColumn("importe", F.round(F.col("cantidad") * F.col("precio_unitario"), 2))
)

pedidos_tx.printSchema()
pedidos_tx.show(5, truncate=False)
print(f"Pedidos validos: {pedidos_tx.count()} (descartados {pedidos.count() - pedidos_tx.count()} por nulos)")


root
 |-- id_pedido: integer (nullable = true)
 |-- id_cliente: integer (nullable = true)
 |-- fecha: date (nullable = true)
 |-- producto: string (nullable = true)
 |-- cantidad: integer (nullable = true)
 |-- precio_unitario: double (nullable = true)
 |-- importe: double (nullable = true)

+---------+----------+----------+---------+--------+---------------+-------+
|id_pedido|id_cliente|fecha     |producto |cantidad|precio_unitario|importe|
+---------+----------+----------+---------+--------+---------------+-------+
|1001     |12        |2025-03-05|Ratón    |6       |16.0           |96.0   |
|1002     |32        |2025-02-02|Ratón    |3       |19.0           |57.0   |
|1003     |5         |2025-03-07|Monitor  |2       |210.0          |420.0  |
|1004     |13        |2025-03-18|Impresora|4       |205.0          |820.0  |
|1005     |41        |2025-03-10|Teclado  |5       |40.0           |200.0  |
+---------+----------+----------+---------+--------+---------------+-------+
only showing t

## 4. Haz un join entre clientes y pedidos.

In [6]:
# Join interno: solo pedidos de clientes existentes
ventas = pedidos_tx.join(clientes_limpio, on="id_cliente", how="inner")
print(f"Filas tras inner join: {ventas.count()}")
ventas.show(5, truncate=False)

# Analisis de registros perdidos
pedidos_huerfanos = pedidos_tx.join(clientes_limpio, on="id_cliente", how="left_anti")
print(f"Pedidos sin cliente (huerfanos): {pedidos_huerfanos.count()}")
pedidos_huerfanos.select("id_pedido", "id_cliente", "producto", "importe").show(truncate=False)

clientes_sin_pedidos = clientes_limpio.join(pedidos_tx, on="id_cliente", how="left_anti")
print(f"Clientes sin ningun pedido: {clientes_sin_pedidos.count()}")
clientes_sin_pedidos.show(truncate=False)


Filas tras inner join: 106
+----------+---------+----------+---------+--------+---------------+-------+------+--------+--------+
|id_cliente|id_pedido|fecha     |producto |cantidad|precio_unitario|importe|nombre|ciudad  |segmento|
+----------+---------+----------+---------+--------+---------------+-------+------+--------+--------+
|12        |1001     |2025-03-05|Ratón    |6       |16.0           |96.0   |Miguel|Zaragoza|Estandar|
|32        |1002     |2025-02-02|Ratón    |3       |19.0           |57.0   |Andres|Zaragoza|Premium |
|5         |1003     |2025-03-07|Monitor  |2       |210.0          |420.0  |Lucia |Bilbao  |Premium |
|13        |1004     |2025-03-18|Impresora|4       |205.0          |820.0  |Irene |Madrid  |Premium |
|27        |1007     |2025-02-13|Ratón    |1       |23.0           |23.0   |Laura |Bilbao  |Estandar|
+----------+---------+----------+---------+--------+---------------+-------+------+--------+--------+
only showing top 5 rows
Pedidos sin cliente (huerfanos)

## 5. Filtra ventas Premium con importe >= 100.

In [7]:
ventas_premium = ventas.filter((F.col("segmento") == "Premium") & (F.col("importe") >= 100))
print(f"Ventas Premium con importe >= 100: {ventas_premium.count()}")
ventas_premium.select("id_pedido", "id_cliente", "nombre", "ciudad", "producto", "importe").show(10, truncate=False)


Ventas Premium con importe >= 100: 34
+---------+----------+------+--------+-----------+-------+
|id_pedido|id_cliente|nombre|ciudad  |producto   |importe|
+---------+----------+------+--------+-----------+-------+
|1003     |5         |Lucia |Bilbao  |Monitor    |420.0  |
|1004     |13        |Irene |Madrid  |Impresora  |820.0  |
|1013     |37        |Oscar |Alicante|Portátil   |3705.0 |
|1014     |16        |Diego |Granada |Webcam     |156.0  |
|1018     |4         |Pablo |Madrid  |Webcam     |159.0  |
|1024     |18        |Jorge |Sevilla |Webcam     |252.0  |
|1026     |37        |Oscar |Alicante|Impresora  |140.0  |
|1027     |22        |Ruben |Granada |Disco SSD  |534.0  |
|1034     |16        |Diego |Granada |Impresora  |545.0  |
|1036     |22        |Ruben |Granada |Auriculares|392.0  |
+---------+----------+------+--------+-----------+-------+
only showing top 10 rows


## 6. Clasifica pedidos con `when` en Alto / Medio / Bajo.

In [8]:
ventas_cls = ventas.withColumn(
    "clasificacion",
    F.when(F.col("importe") >= 500, "Alto")
     .when(F.col("importe") >= 100, "Medio")
     .otherwise("Bajo")
)

ventas_cls.groupBy("clasificacion").count().orderBy(F.desc("count")).show()
ventas_cls.select("id_pedido", "producto", "importe", "clasificacion").show(10, truncate=False)


+-------------+-----+
|clasificacion|count|
+-------------+-----+
|        Medio|   51|
|         Alto|   32|
|         Bajo|   23|
+-------------+-----+

+---------+-----------+-------+-------------+
|id_pedido|producto   |importe|clasificacion|
+---------+-----------+-------+-------------+
|1001     |Ratón      |96.0   |Bajo         |
|1002     |Ratón      |57.0   |Bajo         |
|1003     |Monitor    |420.0  |Medio        |
|1004     |Impresora  |820.0  |Alto         |
|1007     |Ratón      |23.0   |Bajo         |
|1008     |Webcam     |462.0  |Medio        |
|1009     |Teclado    |38.0   |Bajo         |
|1010     |Webcam     |78.0   |Bajo         |
|1011     |Auriculares|64.0   |Bajo         |
|1013     |Portátil   |3705.0 |Alto         |
+---------+-----------+-------+-------------+
only showing top 10 rows


## 7. Calcula agregaciones por ciudad y segmento.

In [9]:
# Metricas por ciudad y segmento
resumen = (
    ventas.groupBy("ciudad", "segmento")
    .agg(
        F.count("id_pedido").alias("num_pedidos"),
        F.round(F.sum("importe"), 2).alias("ingresos_totales"),
        F.round(F.avg("importe"), 2).alias("importe_medio"),
    )
    .orderBy(F.desc("ingresos_totales"))
)
resumen.show(truncate=False)

# Top 5 ciudades por ingresos totales
print("=== Top 5 ciudades por ingresos ===")
(
    ventas.groupBy("ciudad")
    .agg(F.round(F.sum("importe"), 2).alias("ingresos_totales"))
    .orderBy(F.desc("ingresos_totales"))
    .show(5, truncate=False)
)

# Producto mas vendido por cantidad
print("=== Top productos por unidades ===")
(
    ventas.groupBy("producto")
    .agg(F.sum("cantidad").alias("unidades"), F.round(F.sum("importe"), 2).alias("ingresos"))
    .orderBy(F.desc("unidades"))
    .show(truncate=False)
)


+----------+--------+-----------+----------------+-------------+
|ciudad    |segmento|num_pedidos|ingresos_totales|importe_medio|
+----------+--------+-----------+----------------+-------------+
|Bilbao    |Estandar|18         |11610.0         |645.0        |
|Granada   |Premium |7          |9161.0          |1308.71      |
|Alicante  |Estandar|10         |9113.0          |911.3        |
|Alicante  |Premium |7          |7472.0          |1067.43      |
|Bilbao    |Premium |7          |5674.0          |810.57       |
|Sevilla   |Estandar|6          |3091.0          |515.17       |
|Madrid    |Estandar|5          |2719.0          |543.8        |
|Zaragoza  |Estandar|6          |2710.0          |451.67       |
|Madrid    |Premium |5          |2347.0          |469.4        |
|Murcia    |Estandar|7          |2058.0          |294.0        |
|Valencia  |Estandar|6          |1593.0          |265.5        |
|Murcia    |Premium |4          |1390.0          |347.5        |
|Granada   |Estandar|5   

## 8. Crea una vista temporal y haz una consulta SQL.

In [10]:
ventas.createOrReplaceTempView("ventas")

print("=== Ingresos y ticket medio por segmento ===")
spark.sql("""
    SELECT
        segmento,
        COUNT(*)                       AS num_pedidos,
        ROUND(SUM(importe), 2)         AS ingresos_totales,
        ROUND(AVG(importe), 2)         AS ticket_medio
    FROM ventas
    GROUP BY segmento
    ORDER BY ingresos_totales DESC
""").show(truncate=False)

print("=== Top 5 clientes por ingresos ===")
spark.sql("""
    SELECT
        id_cliente,
        nombre,
        ciudad,
        COUNT(*)                AS num_pedidos,
        ROUND(SUM(importe), 2)  AS ingresos
    FROM ventas
    GROUP BY id_cliente, nombre, ciudad
    ORDER BY ingresos DESC
    LIMIT 5
""").show(truncate=False)


=== Ingresos y ticket medio por segmento ===
+--------+-----------+----------------+------------+
|segmento|num_pedidos|ingresos_totales|ticket_medio|
+--------+-----------+----------------+------------+
|Estandar|63         |34022.0         |540.03      |
|Premium |43         |27703.0         |644.26      |
+--------+-----------+----------------+------------+

=== Top 5 clientes por ingresos ===
+----------+------+--------+-----------+--------+
|id_cliente|nombre|ciudad  |num_pedidos|ingresos|
+----------+------+--------+-----------+--------+
|22        |Ruben |Granada |3          |7112.0  |
|37        |Oscar |Alicante|5          |6758.0  |
|10        |David |Bilbao  |5          |6697.0  |
|6         |Carlos|Alicante|6          |5922.0  |
|2         |Luis  |Bilbao  |4          |4973.0  |
+----------+------+--------+-----------+--------+



## 9. Usa `sample()` y `randomSplit()`.

In [11]:
muestra = ventas.sample(withReplacement=False, fraction=0.2, seed=42)
print(f"Sample 20%: {muestra.count()} filas (poblacion: {ventas.count()})")
muestra.show(5, truncate=False)

train, test = ventas.randomSplit([0.8, 0.2], seed=42)
print(f"randomSplit -> train: {train.count()} | test: {test.count()}")


Sample 20%: 19 filas (poblacion: 106)
+----------+---------+----------+---------+--------+---------------+-------+-------+-------+--------+
|id_cliente|id_pedido|fecha     |producto |cantidad|precio_unitario|importe|nombre |ciudad |segmento|
+----------+---------+----------+---------+--------+---------------+-------+-------+-------+--------+
|31        |1010     |2025-02-14|Webcam   |1       |78.0           |78.0   |Beatriz|Granada|Estandar|
|10        |1020     |2025-02-28|Monitor  |1       |255.0          |255.0  |David  |Bilbao |Estandar|
|30        |1023     |2025-02-23|Impresora|2       |192.0          |384.0  |Victor |Bilbao |Estandar|
|10        |1032     |2025-03-14|Portátil |1       |1098.0         |1098.0 |David  |Bilbao |Estandar|
|8         |1043     |2025-02-11|Impresora|1       |186.0          |186.0  |Javier |Madrid |Premium |
+----------+---------+----------+---------+--------+---------------+-------+-------+-------+--------+
only showing top 5 rows
randomSplit -> train

## 10. Guarda el resultado en Parquet en `/opt/spark-apps/salida/resultado_parquet` y léelo de nuevo.

In [12]:
salida = "/opt/spark-apps/salida/resultado_parquet"

(
    ventas_cls
    .write
    .mode("overwrite")
    .partitionBy("segmento")
    .parquet(salida)
)
print(f"Escrito en: {salida}")

# Relectura desde Parquet
ventas_parquet = spark.read.parquet(salida)
print(f"Filas leidas: {ventas_parquet.count()}")
ventas_parquet.printSchema()
ventas_parquet.show(5, truncate=False)


Escrito en: /opt/spark-apps/salida/resultado_parquet
Filas leidas: 106
root
 |-- id_cliente: integer (nullable = true)
 |-- id_pedido: integer (nullable = true)
 |-- fecha: date (nullable = true)
 |-- producto: string (nullable = true)
 |-- cantidad: integer (nullable = true)
 |-- precio_unitario: double (nullable = true)
 |-- importe: double (nullable = true)
 |-- nombre: string (nullable = true)
 |-- ciudad: string (nullable = true)
 |-- clasificacion: string (nullable = true)
 |-- segmento: string (nullable = true)

+----------+---------+----------+-----------+--------+---------------+-------+-------+--------+-------------+--------+
|id_cliente|id_pedido|fecha     |producto   |cantidad|precio_unitario|importe|nombre |ciudad  |clasificacion|segmento|
+----------+---------+----------+-----------+--------+---------------+-------+-------+--------+-------------+--------+
|12        |1001     |2025-03-05|Ratón      |6       |16.0           |96.0   |Miguel |Zaragoza|Bajo         |Estandar|

## 11. Responde brevemente en Markdown:
- ¿Qué ventaja tiene usar join?
- ¿Qué diferencia hay entre sample y randomSplit?
- ¿Qué pedido se pierde en el inner join y por qué?

### Respuestas

- **Ventaja del join**: combina informacion de dos fuentes (clientes + pedidos) para poder analizar las ventas con el contexto del cliente (ciudad, segmento) sin duplicar datos en cada tabla.
- **`sample` vs `randomSplit`**: `sample` devuelve un unico subconjunto de tamano aproximado a la fraccion indicada (con o sin reemplazo); `randomSplit` divide el DataFrame en varias partes disjuntas que suman el total, tipicamente para train/test.
- **Pedidos perdidos en el `inner join`**: los pedidos cuyo `id_cliente` no existe en `clientes` (por ejemplo ids 41, 42, 45, 46, 47, 48 en el dataset) se pierden porque el `inner join` solo conserva coincidencias en ambas tablas.


## 12. Cierra la sesión de Spark.

In [13]:
spark.stop()